In [1]:
from pathlib import Path

from adapt_eeg.data_reader import build_metadata


metadata = build_metadata(Path("../data/raw/itpc-30event-good"))
len(metadata)

700

In [2]:
from adapt_eeg.data_reader import read_unepoch_sample


samples = [read_unepoch_sample(metad) for metad in metadata]

In [ ]:
# reduce all to 2hz
X = [
    sample.data.copy().filter(l_freq=1.8, h_freq=2.2, method="fir", phase="zero")
    for sample in samples
]

In [4]:
from sklearn.model_selection import GroupShuffleSplit


splitter = GroupShuffleSplit(1, test_size=0.2, random_state=0)

groups = [sample.participant for sample in samples]
labels = [sample.rhythm_type for sample in samples]

train_indices, test_indices = next(
    splitter.split(
        X=X,
        y=labels,
        groups=groups,
    )
)

In [5]:
X_train = [X[i] for i in train_indices]
X_test = [X[i] for i in test_indices]

y_train = [labels[i] for i in train_indices]
y_test = [labels[i] for i in test_indices]

# Train a regular-vs-irregular EEG classifier

This notebook builds one feature row per `Sample` returned by `read_unepoch_sample` and trains a regular/irregular classifier.

Important modeling choice: the split is grouped by `participant`, so the same participant never appears in both train and test folds.

In [6]:
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from adapt_eeg.constants import FREQUENCIES
from adapt_eeg.data_reader import CHANNELS_OF_INTEREST, build_metadata, read_unepoch_sample

mne.set_log_level("ERROR")

DATA_ROOT = Path("../data/raw/itpc-30event-good")
RANDOM_STATE = 0

## Feature extraction

`read_unepoch_sample` returns `EpochsArray` with shape `1 x channels x times` for reconstructed samples. The helper below uses the single epoch as `channels x times`; if a future file has multiple epochs, it averages them before feature extraction.

In [7]:
def sample_to_array(data):
    """Return channels x times data and metadata from Raw or 1-epoch EpochsArray."""
    if isinstance(data, mne.BaseEpochs):
        epochs_data = data.get_data(copy=True)
        if epochs_data.shape[0] != 1:
            epochs_data = epochs_data.mean(axis=0, keepdims=True)
        array = epochs_data[0]
        ch_names = data.ch_names
        sfreq = float(data.info["sfreq"])
        times = data.times
    elif isinstance(data, mne.io.BaseRaw):
        array = data.get_data()
        ch_names = data.ch_names
        sfreq = float(data.info["sfreq"])
        times = data.times
    else:
        raise TypeError(f"Unsupported MNE object: {type(data)!r}")

    array = np.asarray(array, dtype=float)
    array = array - array.mean(axis=-1, keepdims=True)
    return array, ch_names, sfreq, times


def band_power(psd, freqs, low, high):
    mask = (freqs >= low) & (freqs < high)
    if not mask.any():
        return np.nan
    return np.trapz(psd[..., mask], freqs[mask], axis=-1)


def frequency_bin_power(psd, freqs, target, width=0.5):
    mask = (freqs >= target - width) & (freqs <= target + width)
    if not mask.any():
        return np.nan
    return np.trapz(psd[..., mask], freqs[mask], axis=-1)


def fourier_phase_strength(array, times, freq):
    window = np.hanning(array.shape[-1])
    kernel = window * np.exp(-2j * np.pi * freq * times)
    coef = np.sum(array * kernel[None, :], axis=-1)
    return np.abs(coef) / np.maximum(np.sum(window), np.finfo(float).eps)


def extract_features(sample):
    array, ch_names, sfreq, times = sample_to_array(sample.data)
    features = {
        "participant": sample.participant,
        "line": sample.line,
        "rhythm_type": sample.rhythm_type.value,
        "language_understanding": sample.language_understanding.value,
        "n_channels": array.shape[0],
        "n_times": array.shape[1],
        "sfreq": sfreq,
        "duration_s": array.shape[1] / sfreq,
    }

    nperseg = min(array.shape[-1], int(round(sfreq * 2)))
    freqs, psd = welch(array, fs=sfreq, nperseg=nperseg, axis=-1)
    total_power = band_power(psd, freqs, 1, 30)

    bands = {
        "delta_1_4": (1, 4),
        "theta_4_8": (4, 8),
        "alpha_8_13": (8, 13),
        "beta_13_30": (13, 30),
    }

    for ch_idx, ch_name in enumerate(ch_names):
        ch = array[ch_idx]
        prefix = ch_name.replace(" ", "_")
        features[f"{prefix}_mean"] = float(ch.mean())
        features[f"{prefix}_std"] = float(ch.std(ddof=1))
        features[f"{prefix}_rms"] = float(np.sqrt(np.mean(ch**2)))
        features[f"{prefix}_ptp"] = float(np.ptp(ch))
        features[f"{prefix}_skew"] = float(stats.skew(ch, bias=False))
        features[f"{prefix}_kurtosis"] = float(stats.kurtosis(ch, bias=False))
        features[f"{prefix}_zero_crossing_rate"] = float(np.mean(np.diff(np.signbit(ch))))

        for band_name, (low, high) in bands.items():
            value = band_power(psd[ch_idx], freqs, low, high)
            features[f"{prefix}_{band_name}_power"] = float(value)
            features[f"{prefix}_{band_name}_relative_power"] = float(
                value / total_power[ch_idx]
            )

        for frequency_name, freq in FREQUENCIES.items():
            power = frequency_bin_power(psd[ch_idx], freqs, freq)
            features[f"{prefix}_{frequency_name.lower()}_{freq:g}hz_power"] = float(power)
            features[f"{prefix}_{frequency_name.lower()}_{freq:g}hz_relative_power"] = float(
                power / total_power[ch_idx]
            )
            features[f"{prefix}_{frequency_name.lower()}_{freq:g}hz_phase_strength"] = float(
                fourier_phase_strength(array[[ch_idx]], times, freq)[0]
            )

    for channel in CHANNELS_OF_INTEREST:
        if channel in ch_names:
            continue
        for suffix in ("mean", "std", "rms", "ptp", "skew", "kurtosis", "zero_crossing_rate"):
            features[f"{channel}_{suffix}"] = np.nan

    return features

## Build the modeling table

One row is one sample. Metadata columns are kept for grouping/reporting, but `participant`, `line`, and labels are excluded from model features to avoid leakage.

In [8]:
metadata = build_metadata(DATA_ROOT)
rows = []
errors = []

for row in metadata:
    try:
        sample = read_unepoch_sample(row)
        rows.append(extract_features(sample))
    except Exception as exc:
        errors.append(
            {
                "participant": row.participant,
                "line": row.line,
                "rhythm_type": row.rhythm_type.value,
                "set_path": row.set_path,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

features = pd.DataFrame(rows)
load_errors = pd.DataFrame(errors)

print(f"Loaded {len(features)} samples; {len(load_errors)} errors")
display(features[["participant", "line", "rhythm_type", "n_channels", "n_times", "sfreq", "duration_s"]].head())
if not load_errors.empty:
    display(load_errors.head())

/tmp/ipykernel_1780636/1692942377.py:28: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[..., mask], freqs[mask], axis=-1)
/tmp/ipykernel_1780636/1692942377.py:35: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[..., mask], freqs[mask], axis=-1)
/tmp/ipykernel_1780636/1692942377.py:28: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[..., mask], freqs[mask], axis=-1)
/tmp/ipykernel_1780636/1692942377.py:35: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[..., mask], freqs[mask], axis=-1)
/tmp/ipykernel_1780636/1692942377.py:28: DeprecationWarning: `trapz`

KeyboardInterrupt: 

In [ ]:
label_col = "rhythm_type"
metadata_cols = {
    "participant",
    "line",
    "rhythm_type",
    "language_understanding",
    "n_channels",
    "n_times",
    "sfreq",
    "duration_s",
}
feature_cols = [col for col in features.columns if col not in metadata_cols]

X = features[feature_cols]
y = features[label_col]
groups = features["participant"]

print(f"Samples: {len(features)}")
print(f"Participants: {groups.nunique()}")
print(f"Features: {len(feature_cols)}")
display(y.value_counts().rename("count"))

Samples: 700
Participants: 50
Features: 399


rhythm_type
regular      350
irregular    350
Name: count, dtype: int64

## Participant-grouped model evaluation

The baseline is a stratified dummy model. The first real model is imputation + standardization + logistic regression.

Because all channels show strong 2 Hz neural entrainment, a PCA model is also evaluated. PCA compresses repeated cross-channel structure before classification. `GroupKFold` prevents participant leakage.

In [ ]:
def evaluate_model(model, X, y, groups, n_splits=5):
    n_splits = min(n_splits, groups.nunique())
    cv = GroupKFold(n_splits=n_splits)
    rows = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            score_regular = model.predict_proba(X_test)[:, list(model.classes_).index("regular")]
        else:
            score_regular = None

        row = {
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_test_participants": groups.iloc[test_idx].nunique(),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "f1_regular": f1_score(y_test, pred, pos_label="regular"),
        }
        if score_regular is not None and y_test.nunique() == 2:
            row["roc_auc_regular"] = roc_auc_score(y_test == "regular", score_regular)
        else:
            row["roc_auc_regular"] = np.nan
        rows.append(row)

    return pd.DataFrame(rows)


dummy_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
    ]
)

logistic_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pca_logistic_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model_specs = {
    "dummy_stratified": dummy_model,
    "logistic_regression": logistic_model,
    "pca_95_logistic_regression": pca_logistic_model,
}

scores = pd.concat(
    [evaluate_model(model, X, y, groups).assign(model=name) for name, model in model_specs.items()],
    ignore_index=True,
)

summary = scores.groupby("model").agg(
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    balanced_accuracy_sd=("balanced_accuracy", "std"),
    f1_regular_mean=("f1_regular", "mean"),
    f1_regular_sd=("f1_regular", "std"),
    roc_auc_regular_mean=("roc_auc_regular", "mean"),
    roc_auc_regular_sd=("roc_auc_regular", "std"),
)

display(scores)
display(summary)

,fold,n_train,n_test,n_test_participants,balanced_accuracy,f1_regular,roc_auc_regular,model
0,1,560,140,10,0.435714,0.414815,0.435714,dummy_stratified
1,2,560,140,10,0.435714,0.414815,0.435714,dummy_stratified
2,3,560,140,10,0.435714,0.414815,0.435714,dummy_stratified
3,4,560,140,10,0.435714,0.414815,0.435714,dummy_stratified
4,5,560,140,10,0.435714,0.414815,0.435714,dummy_stratified
5,1,560,140,10,0.557143,0.515625,0.604286,logistic_regression
6,2,560,140,10,0.614286,0.644737,0.679184,logistic_regression
7,3,560,140,10,0.571429,0.552239,0.631224,logistic_regression
8,4,560,140,10,0.621429,0.595420,0.659388,logistic_regression
9,5,560,140,10,0.564286,0.579310,0.630204,logistic_regression


,balanced_accuracy_mean,balanced_accuracy_sd,f1_regular_mean,f1_regular_sd,roc_auc_regular_mean,roc_auc_regular_sd
model,,,,,,
dummy_stratified,0.435714,0.000000,0.414815,0.000000,0.435714,0.000000
logistic_regression,0.585714,0.029881,0.577466,0.048249,0.640857,0.028966
pca_95_logistic_regression,0.621429,0.041033,0.615780,0.052369,0.670122,0.041044


In [ ]:
pca_logistic_model.fit(X, y)
pca = pca_logistic_model.named_steps["pca"]

print(f"PCA components retained: {pca.n_components_}")
print(f"Explained variance retained: {pca.explained_variance_ratio_.sum():.3f}")

display(
    pd.DataFrame(
        {
            "component": np.arange(1, pca.n_components_ + 1),
            "explained_variance_ratio": pca.explained_variance_ratio_,
            "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
        }
    ).head(25)
)

PCA components retained: 93
Explained variance retained: 0.951


,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.230263,0.230263
1,2,0.114649,0.344912
2,3,0.056211,0.401123
3,4,0.053106,0.454230
4,5,0.048732,0.502962
5,6,0.037208,0.540169
6,7,0.030762,0.570931
7,8,0.027852,0.598783
8,9,0.022465,0.621249
9,10,0.017370,0.638619


In [ ]:
logistic_model.fit(X, y)
classifier = logistic_model.named_steps["classifier"]
coef_regular = classifier.coef_[0]
if classifier.classes_[1] != "regular":
    coef_regular = -coef_regular

coefs = pd.DataFrame(
    {
        "feature": feature_cols,
        "coef_regular": coef_regular,
    }
)
coefs["abs_coef"] = coefs["coef_regular"].abs()

display(coefs.sort_values("abs_coef", ascending=False).head(25))

,feature,coef_regular,abs_coef
55,Pz_beta_13_30_power,1.464664,1.464664
349,T7_beta_13_30_power,1.325287,1.325287
321,Fp1_zero_crossing_rate,1.304366,1.304366
237,Fp2_zero_crossing_rate,-1.193268,1.193268
356,T7_speaking_4hz_phase_strength,-1.153981,1.153981
192,C4_ptp,1.125852,1.125852
279,C3_zero_crossing_rate,1.101610,1.101610
222,F4_alpha_8_13_relative_power,1.067214,1.067214
144,O2_speaking_4hz_power,1.039406,1.039406
162,T8_poem_2hz_power,1.036977,1.036977


In [ ]:
from pathlib import Path
from adapt_eeg.data_reader import build_metadata

metadata = build_metadata(Path("../data/raw/itpc-30event-good"))
len(metadata)

700

In [ ]:
metadata[0]

SampleMetadata(participant=46, line=10, set_path='../data/raw/itpc-30event-good/line10/part46v10.set', fdt_path='../data/raw/itpc-30event-good/line10/part46v10.fdt', rhythm_type=<RhythmType.REGULAR: 'regular'>, language_understanding=<LanguageUnderstanding.YES: 'english_competence'>)